# Deep Reinforcement Learning for Optimal Portfolio Allocation
## A Comparative Study with Mean-Variance Optimization

This notebook implements and compares DRL algorithms (A2C, PPO, DDPG) against traditional Mean-Variance Optimization for portfolio allocation using Dow Jones 30 stocks.

## 1. Install Dependencies

In [ ]:
!pip install -q numpy pandas matplotlib seaborn yfinance gymnasium stable-baselines3 pypfopt ta scikit-learn scipy torch

## 2. Imports and Configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

from config import (
    DOW_30_TICKERS, TRAIN_START_DATE, TRAIN_END_DATE,
    TEST_START_DATE, TEST_END_DATE, TECHNICAL_INDICATORS, INITIAL_AMOUNT
)

print(f'Assets: {len(DOW_30_TICKERS)} Dow Jones stocks')
print(f'Train period: {TRAIN_START_DATE} to {TRAIN_END_DATE}')
print(f'Test period: {TEST_START_DATE} to {TEST_END_DATE}')
print(f'Initial investment: ${INITIAL_AMOUNT:,}')

## 3. Data Download

In [ ]:
from data_downloader import download_data, clean_data, save_data, load_data

data_path = 'datasets/stock_data.csv'
if os.path.exists(data_path):
    print(f'Loading cached data from {data_path}')
    raw_data = load_data(data_path)
else:
    raw_data = download_data()
    raw_data = clean_data(raw_data)
    save_data(raw_data)

print(f'\nData shape: {raw_data.shape}')
print(f'Date range: {raw_data["date"].min()} to {raw_data["date"].max()}')
print(f'Tickers: {raw_data["tic"].nunique()}')
raw_data.head()

### Exploratory Data Analysis

In [ ]:
# Plot adjusted close prices for a sample of stocks
fig, ax = plt.subplots(figsize=(14, 7))
sample_tickers = ['AAPL', 'MSFT', 'JPM', 'JNJ', 'BA']
for tic in sample_tickers:
    subset = raw_data[raw_data['tic'] == tic]
    prices = subset['adjclose'].values / subset['adjclose'].values[0]  # Normalize
    ax.plot(subset['date'].values, prices, label=tic, linewidth=1.5)

ax.set_title('Normalized Stock Prices (Sample)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Price')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
from feature_engineering import prepare_features

features_path = 'datasets/features_data.csv'
if os.path.exists(features_path):
    print(f'Loading cached features from {features_path}')
    data = pd.read_csv(features_path, parse_dates=['date'])
else:
    data = prepare_features(raw_data)
    os.makedirs('datasets', exist_ok=True)
    data.to_csv(features_path, index=False)

print(f'Features shape: {data.shape}')
print(f'Columns: {list(data.columns[:20])}...')
data.head()

## 5. Train/Test Split

In [ ]:
train_data = data[data['date'] < TEST_START_DATE].reset_index(drop=True)
test_data = data[data['date'] >= TEST_START_DATE].reset_index(drop=True)
tickers = sorted(data['tic'].unique())

print(f'Train: {train_data["date"].min()} to {train_data["date"].max()} ({train_data["date"].nunique()} days)')
print(f'Test:  {test_data["date"].min()} to {test_data["date"].max()} ({test_data["date"].nunique()} days)')
print(f'Number of tickers: {len(tickers)}')

## 6. Build Environment and Train DRL Models

In [ ]:
from env_portfolio import PortfolioAllocationEnv
from models import get_model, train_model, save_model, load_model, predict_with_model


def build_env(df, tickers):
    stock_dim = len(tickers)
    n_indicators = len(TECHNICAL_INDICATORS)
    cov_columns = [c for c in df.columns if c.startswith('cov_')]
    n_cov = len(cov_columns)
    state_space = stock_dim + stock_dim + n_indicators * stock_dim + n_cov
    action_space = stock_dim
    return PortfolioAllocationEnv(
        df=df, stock_dim=stock_dim,
        state_space=state_space, action_space=action_space
    )


train_env = build_env(train_data, tickers)
print(f'State space: {train_env.observation_space.shape}')
print(f'Action space: {train_env.action_space.shape}')

In [ ]:
# Train A2C
algorithms = ['a2c', 'ppo', 'ddpg']
trained_models = {}

for algo in algorithms:
    model_path = f'trained_models/{algo}.zip'
    if os.path.exists(model_path):
        print(f'Loading pre-trained {algo.upper()} model...')
        model = load_model(algo, train_env)
    else:
        model = get_model(algo, train_env)
        model = train_model(model, algo)
        save_model(model, algo)
    trained_models[algo] = model
    print()

## 7. Compute Benchmark Portfolios

In [ ]:
from benchmarks import equal_weight_portfolio, max_sharpe_portfolio

ew_df = equal_weight_portfolio(test_data)
print(f'Equal Weight final value: ${ew_df["portfolio_value"].iloc[-1]:,.2f}\n')

mvo_df = max_sharpe_portfolio(train_data, test_data)
print(f'Max Sharpe (MVO) final value: ${mvo_df["portfolio_value"].iloc[-1]:,.2f}')

## 8. Evaluate DRL Models on Test Set

In [ ]:
results = {}

for algo in algorithms:
    test_env = build_env(test_data, tickers)
    model = trained_models[algo]
    model.set_env(test_env)
    portfolio_df = predict_with_model(model, test_env)
    results[algo.upper()] = portfolio_df
    print(f'{algo.upper()}: Final value = ${portfolio_df["portfolio_value"].iloc[-1]:,.2f}')

results['Equal Weight'] = ew_df
results['Max Sharpe (MVO)'] = mvo_df

## 9. Performance Comparison

In [ ]:
from backtest import compare_strategies, print_summary

summary = compare_strategies(results)
print_summary(summary)
summary

## 10. Visualizations

In [ ]:
# Cumulative Returns
fig, ax = plt.subplots(figsize=(14, 7))
for name, df in results.items():
    dates = pd.to_datetime(df['date'])
    values = df['portfolio_value'].values
    normalized = values / values[0]
    ax.plot(dates, normalized, label=name, linewidth=2)

ax.set_title('Cumulative Portfolio Returns Comparison', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Portfolio Value')
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Drawdown Comparison
fig, ax = plt.subplots(figsize=(14, 5))
for name, df in results.items():
    dates = pd.to_datetime(df['date'])
    values = df['portfolio_value'].values
    peak = np.maximum.accumulate(values)
    drawdown = (peak - values) / peak * 100
    ax.plot(dates, -drawdown, label=name, linewidth=1.5)

ax.set_title('Portfolio Drawdown Comparison', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Drawdown (%)')
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Metrics Bar Charts
metrics_to_plot = ['Cumulative Return (%)', 'Annualized Return (%)', 'Sharpe Ratio', 'Max Drawdown (%)']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for idx, metric in enumerate(metrics_to_plot):
    ax = axes.flatten()[idx]
    values = summary[metric]
    bars = ax.bar(values.index, values.values, color=plt.cm.tab10(np.arange(len(values))))
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    for bar, val in zip(bars, values.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{val:.2f}',
                ha='center', va='bottom', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Strategy Performance Metrics', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Daily Returns Distribution
fig, axes = plt.subplots(1, len(results), figsize=(4*len(results), 5), sharey=True)
for ax, (name, df) in zip(axes, results.items()):
    returns = df['daily_return'].values[1:]
    ax.hist(returns, bins=50, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.axvline(x=np.mean(returns), color='red', linestyle='--',
               label=f'Mean: {np.mean(returns):.4f}')
    ax.set_title(name)
    ax.set_xlabel('Daily Return')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Frequency')
plt.suptitle('Daily Returns Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Save Results

In [ ]:
from visualize import generate_all_plots

# Save all plots to results/
generate_all_plots(results, summary)

# Save summary CSV
os.makedirs('results', exist_ok=True)
summary.to_csv('results/performance_summary.csv')
print('\nAll results saved!')
print('Summary:')
summary